# Notebook 02 — Preprocessing

Giữ nguyên cột `pm2.5` gốc. Không nội suy tuyến tính để tạo nhãn/mục tiêu ML. Các bước bên dưới chỉ chuẩn hóa timestamp, sắp xếp dữ liệu, thêm cờ missing và mã hóa hướng gió; dữ liệu PM2.5 quan sát gốc vẫn được bảo toàn.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

# Tìm gốc repo có chứa file dữ liệu gốc để đường dẫn chạy được trên Colab.
DATA_RELATIVE = Path("PRSA_data_2010.1.1-2014.12.31.csv")
repo_candidates = [
    Path.cwd(),
    Path("/content/PPNCKH_5Bros"),
    Path("/content"),
]
PROJECT_ROOT = next(
    (root.resolve() for root in repo_candidates if (root / DATA_RELATIVE).exists()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu gốc: {DATA_RELATIVE}. "
        "Kiểm tra repo đã được clone vào /content/PPNCKH_5Bros và file raw nằm đúng vị trí."
    )

DATA_PATH = PROJECT_ROOT / DATA_RELATIVE
OUTPUT_ROOT = PROJECT_ROOT / "PPNCKH_5Bros/notebooks/phuong_nb/output"
DATA_OUT = OUTPUT_ROOT / "data"
FIG_OUT = OUTPUT_ROOT / "figures"
TABLE_OUT = OUTPUT_ROOT / "tables"
for folder in (DATA_OUT, FIG_OUT, TABLE_OUT):
    folder.mkdir(parents=True, exist_ok=True)

def load_raw_data():
    data = pd.read_csv(DATA_PATH)
    required = {"year", "month", "day", "hour"}
    missing_cols = required - set(data.columns)
    if missing_cols:
        raise ValueError(f"Thiếu cột thời gian: {sorted(missing_cols)}")
    data["datetime"] = pd.to_datetime(data[["year", "month", "day", "hour"]], errors="coerce")
    return data

print("Repo root:", PROJECT_ROOT)
print("Data file:", DATA_PATH)
print("Output root:", OUTPUT_ROOT)

Repo root: /content
Data file: /content/PRSA_data_2010.1.1-2014.12.31.csv
Output root: /content/PPNCKH_5Bros/notebooks/phuong_nb/output


In [2]:
df = load_raw_data()

# Giữ cột gốc và chỉ chuẩn hóa thứ tự thời gian.
df = df.loc[df["datetime"].notna()].copy()
df = df.sort_values("datetime")
df = df.set_index("datetime")
df.index.name = "datetime"

# Giữ nguyên pm2.5; không điền missing vào cột mục tiêu.
if "pm2.5" not in df.columns:
    raise ValueError("Không tìm thấy cột pm2.5 trong dữ liệu gốc.")

df["pm2.5_missing_flag"] = df["pm2.5"].isna().astype("int8")

# Mã hóa hướng gió nếu cột tồn tại; không thay đổi PM2.5.
if "cbwd" in df.columns:
    df = pd.get_dummies(df, columns=["cbwd"], prefix="wind", dtype=int)

# Export preprocessing data. Lưu ý: vẫn giữ các hàng có PM2.5 missing.
processed_path = DATA_OUT / "beijing_preprocessed.csv"
df.to_csv(processed_path)
print("Rows retained:", len(df))
print("Original PM2.5 missing:", int(df["pm2.5"].isna().sum()))
print("Saved:", processed_path)
display(df.head())

Rows retained: 43824
Original PM2.5 missing: 2067
Saved: /content/PPNCKH_5Bros/notebooks/phuong_nb/output/data/beijing_preprocessed.csv


,No,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,Iws,Is,Ir,pm2.5_missing_flag,wind_NE,wind_NW,wind_SE,wind_cv
datetime,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,1,2010,1,1,0,NaN,-21,-11.0,1021.0,1.79,0,0,1,0,1,0,0
2010-01-01 01:00:00,2,2010,1,1,1,NaN,-21,-12.0,1020.0,4.92,0,0,1,0,1,0,0
2010-01-01 02:00:00,3,2010,1,1,2,NaN,-21,-11.0,1019.0,6.71,0,0,1,0,1,0,0
2010-01-01 03:00:00,4,2010,1,1,3,NaN,-21,-14.0,1019.0,9.84,0,0,1,0,1,0,0
2010-01-01 04:00:00,5,2010,1,1,4,NaN,-20,-12.0,1018.0,12.97,0,0,1,0,1,0,0


**Quy ước:** `pm2.5` là giá trị quan sát gốc và là nguồn tạo target. Không tạo `pm2.5_clean` bằng nội suy để tránh biến dữ liệu ước lượng thành nhãn dự báo. Khi huấn luyện, xử lý missing phải được thực hiện trong pipeline và chỉ dựa trên thông tin có sẵn tại thời điểm dự báo.